In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [ ]:
# Data Ingestion -: From Website we load data using Beautiful Soup4
from langchain_community.document_loader import WebBasedLoader
loader = WebBasedLoader("https://docs.langchain.com/langsmith/observability-llm-tutorial")
loader

In [ ]:
docs=loader.load()
docs

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [ ]:
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings()

In [ ]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [ ]:
vectorstoredb

In [ ]:
query="Ask Any Query"
result=vectorstoredb.similarity_search(query)
result[0].page_content

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")
print(llm)

In [ ]:
# Retrieval chain
from langchain.chains.combine_documents import create_stuff_document_chain
from langchain_core.prompts import ChatPromptsTemplate

prompt=ChatPromptTemplate.from_message(
    """
    Answer the following question based only on the provided context:
    <context>
    {context}
    </context>"""
)
# document chain will provide the context
document_chain=create_stuff_document_chain(llm,prompt)
document_chain


In [ ]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"What is Langsmith?",
    "context":[Document(page_content="Langsmith has two usage, add other data for the information")]
})

In [ ]:
# Retriever

retriever=vectorstoredb.as_retriever()
from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever, document_chain)

In [ ]:
# Get the response from the LLM
retrieval_chain.invoke({
    "input":"What is Langsmith?"
})